In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import librosa
import numpy as np
import pandas as pd
import os
import ssl
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ssl._create_default_https_context = ssl._create_unverified_context

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [2]:
N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 41

In [3]:
clip_df = pd.read_csv('../clips_metadata.csv')
print(f'Total clips: {len(clip_df)}')
print(f'Species: {clip_df["species"].nunique()}')

le = LabelEncoder()
clip_df['label'] = le.fit_transform(clip_df['species'])
print(f'Classes: {len(le.classes_)}')

Total clips: 49682
Species: 41
Classes: 41


In [4]:
class AviaNetDataset(Dataset):
    def __init__(self, df, augment_data=False):
        self.df = df.reset_index(drop=True)
        self.augment_data = augment_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        try:
            y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
        except Exception:
            y = np.zeros(SR * DURATION)
            sr = SR

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

    def augment(self, y, sr):
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
        shift = np.random.randint(0, max(1, sr // 4))
        y = np.roll(y, shift)
        return y

print('Dataset class defined')

Dataset class defined


In [5]:
train_df, test_df = train_test_split(
    clip_df, test_size=0.2, random_state=42, stratify=clip_df['label']
)

train_dataset = AviaNetDataset(train_df, augment_data=True)
test_dataset  = AviaNetDataset(test_df,  augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 39745
Test samples:  9937
Train batches: 622
Test batches:  156
Batch X shape: torch.Size([64, 1, 128, 216])
Batch y shape: torch.Size([64])


In [6]:
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')

# adapt for 1-channel grayscale spectrograms
efficientnet.features[0][0] = nn.Conv2d(
    1, 32, kernel_size=3, stride=2, padding=1, bias=False
)

# replace classifier for 41 species
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

efficientnet = efficientnet.to(device)

# freeze all layers
for param in efficientnet.parameters():
    param.requires_grad = False

# unfreeze classifier + first conv only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet loaded and frozen')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet loaded and frozen
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=41, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_p1 = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_p1 = optim.lr_scheduler.StepLR(optimizer_p1, step_size=10, gamma=0.5)

EPOCHS_P1 = 30
p1_losses = []
p1_accuracies = []

for epoch in range(EPOCHS_P1):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_p1.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_p1.step()
        running_loss += loss.item()

    scheduler_p1.step()

    efficientnet.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    p1_losses.append(avg_loss)
    p1_accuracies.append(acc)
    print(f'Epoch [{epoch+1}/{EPOCHS_P1}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

In [ ]:
for param in efficientnet.parameters():
    param.requires_grad = True

optimizer_p2 = optim.Adam(efficientnet.parameters(), lr=0.0001)
scheduler_p2 = optim.lr_scheduler.StepLR(optimizer_p2, step_size=10, gamma=0.5)

EPOCHS_P2 = 20
p2_losses = []
p2_accuracies = []

for epoch in range(EPOCHS_P2):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_p2.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_p2.step()
        running_loss += loss.item()

    scheduler_p2.step()

    efficientnet.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    p2_losses.append(avg_loss)
    p2_accuracies.append(acc)
    print(f'Epoch [{epoch+31}/{50}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

In [ ]:
all_losses = p1_losses + p2_losses
all_accuracies = p1_accuracies + p2_accuracies

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_losses)
plt.axvline(x=30, color='r', linestyle='--', label='Unfreeze point')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(all_accuracies)
plt.axvline(x=30, color='r', linestyle='--', label='Unfreeze point')
plt.axhline(y=0.6178, color='g', linestyle='--', label='SVM baseline (61.78%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Test Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('../models', exist_ok=True)
torch.save(efficientnet.state_dict(), '../models/avianet_efficientnet.pth')
print('Model saved to ../models/avianet_efficientnet.pth')